# R4 — Metric views (shared definitions, defined once)

The whole complaint behind this paper is three tools giving three different numbers. These metric views define the four numbers people actually argue about — **claim count, approved amount, rejection rate, average turnaround** — exactly once, so a dashboard, Genie, or a one-off query *reads* the definition instead of re-deriving it.

* `claims.analytics.mv_claims` — claims measures, groupable by plan_code, city_tier, hospital_id, claim_type, admission_month.
* `claims.analytics.mv_loss_ratio` — loss-ratio measures, built from the **monthly figures w01 already produced** (never recomputed from scratch), groupable by plan_code, month_start.
* **Rejection rate = rejected ÷ DECIDED** (approved + rejected + settled). A claim still at `submitted` is on neither side of the fraction — the definition lives inside the measure, so no downstream tool can get it wrong.
* If metric views are unavailable on this workspace (Free Edition public preview), the permitted **fallback** at the end builds `metric_definitions` + `v_metrics_by_plan` to the delivery-contract shape.

In [0]:
%sql
-- mv_claims: the four argued-about numbers, defined once (source: the trusted fact + dims)
CREATE OR REPLACE VIEW claims.analytics.mv_claims
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: claims.analytics.fact_claims
  comment: Claims metrics defined once for every downstream tool
  joins:
    - name: plan
      source: claims.analytics.dim_plan
      on: source.plan_sk = plan.plan_sk
    - name: hospital
      source: claims.analytics.dim_hospital
      on: source.hospital_sk = hospital.hospital_sk
    - name: dates
      source: claims.analytics.dim_date
      on: source.admission_date_key = dates.date_key
  dimensions:
    - name: plan_code
      expr: plan.plan_code
    - name: city_tier
      expr: hospital.city_tier
    - name: hospital_id
      expr: hospital.hospital_id
    - name: claim_type
      expr: source.claim_type
    - name: admission_month
      expr: dates.month_start
  measures:
    - name: claim_count
      expr: COUNT(1)
    - name: approved_amount
      expr: SUM(source.approved_amount)
    - name: rejection_rate
      expr: |-
        SUM(CASE WHEN source.status = 'rejected' THEN 1 ELSE 0 END)
        / SUM(CASE WHEN source.status IN ('approved', 'rejected', 'settled') THEN 1 ELSE 0 END)
      comment: Rejected over DECIDED claims — a submitted claim is on neither side
    - name: avg_turnaround_days
      expr: AVG(source.turnaround_days)
$$;

In [0]:
%sql
-- mv_loss_ratio: built from the monthly figures w01 already produced (not recomputed from scratch)
CREATE OR REPLACE VIEW claims.analytics.mv_loss_ratio
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: claims.analytics.w01
  comment: Loss-ratio metrics from the monthly w01 figures
  dimensions:
    - expr: plan_code
    - expr: month_start
  measures:
    - name: paid_amount
      expr: SUM(paid_amount)
    - name: earned_premium
      expr: SUM(earned_premium)
    - name: loss_ratio
      expr: MEASURE(paid_amount) / MEASURE(earned_premium)
$$;

In [0]:
%sql
-- Read the shared definitions: the four numbers per plan (as a dashboard or Genie would)
SELECT
  plan_code,
  MEASURE(claim_count)         AS claim_count,
  MEASURE(approved_amount)     AS approved_amount,
  MEASURE(rejection_rate)      AS rejection_rate,
  MEASURE(avg_turnaround_days) AS avg_turnaround_days
FROM claims.analytics.mv_claims
GROUP BY plan_code
ORDER BY plan_code;

In [0]:
%sql
-- Loss ratio per plan from the shared view (rejection_rate must equal rejected / DECIDED)
SELECT
  plan_code,
  MEASURE(paid_amount)   AS paid_amount,
  MEASURE(earned_premium) AS earned_premium,
  MEASURE(loss_ratio)    AS loss_ratio
FROM claims.analytics.mv_loss_ratio
GROUP BY plan_code
ORDER BY plan_code;

In [0]:
%sql
-- PERMITTED FALLBACK (only if metric views are unavailable on this workspace):
-- metric_definitions + v_metrics_by_plan to the delivery-contract shape.
CREATE OR REPLACE TABLE claims.analytics.metric_definitions (
  metric_name STRING,
  expression  STRING,
  source      STRING,
  description STRING
);

INSERT INTO claims.analytics.metric_definitions VALUES
  ('claim_count',    'COUNT(*)',                                                                                   'claims.analytics.fact_claims', 'All claims regardless of status'),
  ('approved_amount','SUM(approved_amount)',                                                                       'claims.analytics.fact_claims', 'Approved amount across all claims'),
  ('rejection_rate', 'SUM(CASE WHEN status=''rejected'' THEN 1 ELSE 0 END) / SUM(CASE WHEN status IN (''approved'',''rejected'',''settled'') THEN 1 ELSE 0 END)', 'claims.analytics.fact_claims', 'Rejected over DECIDED claims — a submitted claim is on neither side'),
  ('avg_turnaround_days','AVG(turnaround_days)',                                                                   'claims.analytics.fact_claims', 'Average days from submission to decision'),
  ('loss_ratio',     'SUM(paid_amount) / SUM(earned_premium)',                                                     'claims.analytics.w01',         'Paid in settled month / earned premium, per plan');

CREATE OR REPLACE VIEW claims.analytics.v_metrics_by_plan AS
SELECT
  p.plan_code,
  COUNT(*)                                                                  AS claim_count,
  CAST(ROUND(SUM(f.approved_amount), 2) AS DECIMAL(14,2))                    AS approved_amount,
  CAST(ROUND(SUM(CASE WHEN f.status='rejected' THEN 1 ELSE 0 END)
        / SUM(CASE WHEN f.status IN ('approved','rejected','settled') THEN 1 ELSE 0 END), 4) AS DECIMAL(10,4)) AS rejection_rate,
  CAST(ROUND(AVG(f.turnaround_days), 2) AS DECIMAL(10,2))                    AS avg_turnaround_days,
  CAST(ROUND(AVG(t.loss_ratio), 2) AS DECIMAL(10,2))                         AS loss_ratio
FROM claims.analytics.fact_claims f
JOIN claims.analytics.dim_plan p ON f.plan_sk = p.plan_sk
LEFT JOIN claims.analytics.w01 t ON p.plan_code = t.plan_code
GROUP BY p.plan_code
ORDER BY p.plan_code;

SELECT * FROM claims.analytics.v_metrics_by_plan;